# Lab 7: Linear Regression Modeling and Interpretation

We train and compare OLS, Ridge, and Lasso models and interpret coefficients.

## Step 1 - Load Cleaned Data and Compare Models
Train OLS, Ridge, and Lasso and save comparison metrics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"
PROCESSED_DIR = Path("..") / ".." / "data" / "processed"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "lab4_train_model_ready.csv"
TEST_PATH = PROCESSED_DIR / "lab4_test_model_ready.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing cleaned dataset: {TRAIN_PATH.resolve()}")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None

features = ["trust", "gdp_per_capita", "life_expectancy", "freedom", "social_support", "generosity"]
target = "happiness_score"

frames = [train_df]
if test_df is not None:
    frames.append(test_df)
df_model = pd.concat(frames, ignore_index=True)

if test_df is None:
    X = df_model[features]
    y = df_model[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
else:
    X = df_model[features]
    y = df_model[target]
    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

models = {
    "OLS": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Lasso": LassoCV(alphas=np.logspace(-3, 0, 30), cv=5, max_iter=50000),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "model": name,
        "mae": mean_absolute_error(y_test, preds),
        "rmse": np.sqrt(mean_squared_error(y_test, preds)),
        "r2": r2_score(y_test, preds),
    })

results_df = pd.DataFrame(results).sort_values(by="r2", ascending=False)
results_path = TABLES_DIR / "lab7_model_comparison.csv"
results_df.to_csv(results_path, index=False)

results_df

print(f"Selected Ridge alpha: {models['Ridge'].alpha_:.4f}")
print(f"Selected Lasso alpha: {models['Lasso'].alpha_:.4f}")

In [ ]:
from sklearn.linear_model import lasso_path

alphas, coefs, _ = lasso_path(X_train, y_train, alphas=np.logspace(-2, 1, 50))

plt.figure(figsize=(9, 5))
for i, feat in enumerate(features):
    plt.plot(alphas, coefs[i], label=feat)
plt.xscale("log")
plt.xlabel("alpha (regularization strength)")
plt.ylabel("coefficient")
plt.title("Lasso Coefficient Path")
plt.legend(loc="upper right", fontsize=8)
plt.gca().invert_xaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab7_lasso_path.png", dpi=300)
plt.show()

Observation: "The Lasso path shows generosity shrinks to zero first (at alpha ≈ X), followed by trust. GDP and social support are the last features standing — confirming our coefficient ranking is robust, not just a quirk of OLS."


## Step 2 - Coefficient Interpretation (OLS)

In [ ]:
import statsmodels.api as sm
X_train_sm = sm.add_constant(X_train)
ols_sm = sm.OLS(y_train, X_train_sm).fit()
print(ols_sm.summary())

**Observation:**
With statsmodels, I can see not just which features have large coefficients, but also which ones are statistically distinguishable from zero.


## Step 3 - Residuals (Best Model)

In [ ]:
best_model = models[results_df.iloc[0]["model"]]
preds = best_model.predict(X_test)
residuals = y_test - preds

plt.figure(figsize=(6, 4))
sns.scatterplot(x=preds, y=residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title(f"Residuals vs Predicted ({results_df.iloc[0]['model']})")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab7_plot_residuals.png", dpi=300)
plt.show()